# E4 — BIO span head + CLS head, jointly trained (REVISION_PLAN E4 / old Exp07)

Systems D/E/F pair mBERT with QA-style start/end span heads. System G is a BIO tagger with **no classifier at all** (structurally penalized on Joint F1 — see `key_numbers.md`). E4 builds the missing cell: mBERT + BIO span head + CLS head, **jointly trained** (`run_17_bio_cls_joint.py`), loss weights identical to System E's (`cls=0.3`, `bio=1.9`, matching `Train_Join.py` exactly) so the only variable changed is head architecture. Saves predictions in `Train_Join.py`'s schema → registers via `Full_evaluation.py --joint_preds`, directly comparable to System D/E's Joint F1.

**If E4 ≈ D/E:** simpler BIO architecture, no QA pointer needed, at no Joint-F1 cost.
**If E4 < D/E:** QA-style span heads earn their complexity.

**Persistence:** all outputs are symlinked to Drive by the runner; it hard-fails before training if a dir is not Drive-backed. If the session times out, **just re-run the training cell** — finished seeds are skipped, it resumes at the first incomplete one.

Run cells top to bottom. Use a **GPU** runtime (Runtime → Change runtime type → T4/A100).

In [ ]:
# 1. Config
REPO_URL = 'https://github.com/JustLetMeBeHello/Idiomator_Research.git'
BRANCH   = 'main'
REPO     = '/content/Idiomator_Research'   # absolute — never use a relative %cd
SEEDS    = '42 123 7'                       # set to e.g. '42' to run one seed
FORCE    = '0'                              # '1' retrains even completed seeds
DRIVE_OUT = '/content/drive/MyDrive/IdiomatorRigor'
print('repo:', REPO, '| seeds:', SEEDS, '| force:', FORCE)

In [ ]:
# 2. Clone / refresh repo, pin absolute cwd, kill any nested duplicate clone
import os, subprocess, sys
from pathlib import Path
# guard against the /content/Idiomator_Research/Idiomator_Research nesting bug
nested = os.path.join(REPO, 'Idiomator_Research')
if os.path.isdir(nested):
    subprocess.run(['rm', '-rf', nested], check=True)
if os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '--quiet', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '--quiet', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--quiet', '--branch', BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)   # ABSOLUTE — the runner self-cds to git toplevel from here
print('cwd:', os.getcwd())
print('head:', subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip())
assert Path('experiments/rigor/run_17b_bio_cls_joint.sh').exists(), 'runner missing — wrong repo/branch? did you push run_17?'

In [ ]:
# 3. Install deps + confirm GPU
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'Requirements.txt'], check=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU — switch runtime to T4/A100'

In [ ]:
# 4. Mount Drive + export env the runner reads
from google.colab import drive
drive.mount('/content/drive')
Path(DRIVE_OUT).mkdir(parents=True, exist_ok=True)
os.environ['DRIVE_OUT'] = DRIVE_OUT
os.environ['SEEDS']     = SEEDS
os.environ['FORCE']     = FORCE
print('DRIVE_OUT =', DRIVE_OUT)

In [ ]:
# 5. GPU smoke test (~1-2 min): 1 epoch, English only, throwaway dir — catches
#    runtime/CUDA issues before the long run. Trains nothing that's kept.
#    (Already verified locally on CPU/MPS before this notebook was written —
#    this cell is the GPU-specific re-check, same convention as E3's notebook.)
!python experiments/rigor/run_17_bio_cls_joint.py \
    --output_dir /tmp/_e4_smoke --langs English --test_langs English \
    --epochs 1 --batch_size 8

In [ ]:
# 6. Dry-run the real runner: prints the plan + persistence gate, trains nothing
!bash experiments/rigor/run_17b_bio_cls_joint.sh --dry-run

In [ ]:
# 7. FULL E4 RUN — 3 seeds, Drive-gated, resumable.
#    Re-run this cell after any timeout: finished seeds skip, resumes the rest.
#    console.log streams to Drive (tee -a) so a partial log survives a kill.
LOG = f'{DRIVE_OUT}/e4_bio_cls_joint_console.log'
!bash experiments/rigor/run_17b_bio_cls_joint.sh 2>&1 | tee -a "$LOG"
print('\nlog →', LOG)

In [ ]:
# 8. Persistence readback + 3-seed summary — reads metrics.json back FROM DRIVE
#    (not /content) so this confirms the results actually persisted. Flags any
#    seed whose best epoch hit the cap (the E1 undertraining failure mode) and
#    prints the Joint F1 (geomean of cls_macro_f1 and span_overlap) used for
#    checkpoint selection — same convention as System E.
import json
EPOCH_CAP = 7   # run_17 default; best_epoch == cap ⇒ possibly not converged
print(f"{'seed':<6}{'test_cls_f1':<14}{'test_exact':<12}{'test_ovlp_f1':<14}{'best_epoch':<12}{'best_dev_joint_f1':<18}{'persisted?'}")
rows = []
for seed in SEEDS.split():
    mp = Path(DRIVE_OUT) / f'bio_cls_joint_mbert_s{seed}' / 'metrics.json'
    if not mp.exists():
        print(f'{seed:<6}— metrics.json NOT on Drive (incomplete / not persisted)')
        continue
    M = json.load(open(mp)); rows.append((seed, M))
    flag = '  ⚠ at cap — check convergence' if M['best_epoch'] >= EPOCH_CAP else ''
    print(f"{seed:<6}{M['test_cls_macro_f1']:<14}{M['test_span_exact']:<12}{M['test_span_overlap']:<14}{M['best_epoch']:<12}{M['best_dev_joint_f1']:<18}yes{flag}")
if len(rows) == len(SEEDS.split()) and rows:
    import statistics as st
    clsf1 = [m['test_cls_macro_f1'] for _, m in rows]
    ex = [m['test_span_exact'] for _, m in rows]
    ov = [m['test_span_overlap'] for _, m in rows]
    mean_sd = lambda v: (round(st.mean(v), 4), round(st.pstdev(v), 4))
    print(f"\n3-seed test cls F1   mean±sd: {mean_sd(clsf1)}")
    print(f"3-seed test exact    mean±sd: {mean_sd(ex)}")
    print(f"3-seed test ovlpF1   mean±sd: {mean_sd(ov)}")
    print('\nNext: register each seed (Full_evaluation --joint_preds, cmds printed by the runner),')
    print('then compare Joint F1 directly against System D/E — the "simpler BIO architecture,')
    print('no QA pointer needed" test (E4).')